# Defect detection pipeline

Does super-resolving a low-resolution capture improve automated defect detection? Every method reconstructs the same LR test split at the HR frame size, and the reconstructions are classified with the fine-tuned VGG16.

Thirteen rows are compared: the LR baseline, the HR ceiling, the three learned models (SRCNN, EDSR, ESRGAN), the four interpolations and the four advanced classic algorithms. The eleven reconstructions all produce RGB images at 478x478, which is the regime the HR classifier was trained on.

The two reference rows bracket the comparison and neither is reconstructed. The LR baseline is the only row scored with the LR-trained classifier, at its native resolution, so comparing an SR row against it isolates the effect of the reconstruction rather than a change of classifier input size. The HR row is the original frame scored by the same classifier that scores every reconstruction, which is what turns an absolute accuracy into a readable one: a method at 91 % means something different if HR reaches 93 % than if it reaches 99 %.

Requires `4_srcnn`, `5_edsr`, `6_esrgan` and `8_vgg16` to have been run.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from srlib.constants import CLASS_LABELS_PATH, HR_ROOT, LR_ROOT
from srlib.dataset.loading import load_defect_detection_pipeline_dataset
from srlib.defect_detection.pipeline import DefectDetectionPipeline
from srlib.model_registry import print_available_runs

## Load the test split

The same split the classifier was trained against, so the test images were never seen during training.

In [ ]:
(
    X_LR_train, X_HR_train, y_train,
    X_LR_val, X_HR_val, y_val,
    X_LR_test, X_HR_test, y_test,
) = load_defect_detection_pipeline_dataset(HR_ROOT, LR_ROOT, CLASS_LABELS_PATH)

print(f"LR splits -> train: {X_LR_train.shape}, val: {X_LR_val.shape}, test: {X_LR_test.shape}")
print(f"HR splits -> train: {X_HR_train.shape}, val: {X_HR_val.shape}, test: {X_HR_test.shape}")
print(f"y splits  -> train: {y_train.shape}, val: {y_val.shape}, test: {y_test.shape}")

## Run the pipeline

Leave a run as `None` to take the most recent checkpoint of that model, or pin a timestamp to reproduce an older comparison.

In [ ]:
print_available_runs()

In [ ]:
pipeline = DefectDetectionPipeline(
    X_LR_test=X_LR_test,
    X_HR_test=X_HR_test,
    y_test=y_test,
    srcnn_run=None,
    edsr_run=None,
    esrgan_run=None,
    vgg16_run=None,
)

sr_images, labels, confidences = pipeline.run()

## Confusion matrices

One matrix per method, in evaluation order.

In [ ]:
pipeline.plot_confusion_matrices()

## Classification report

Accuracy, macro recall and both F1 aggregates, plus the same scores resolved per class. The dataset is imbalanced, so macro F1 is the figure to read rather than accuracy.

In [ ]:
fig, axes, report_metrics = pipeline.plot_classification_reports()

## Confidence

Global mean confidence, the same split into correct and wrong predictions, and the error rate. The three together are what tell whether a method is confidently wrong.

In [ ]:
fig, axes, confidence_metrics = pipeline.plot_confidence()

## Qualitative comparison

Every reconstruction of a single test image with the prediction it produced. Change `index` to inspect another sample.

In [ ]:
pipeline.plot_sample_predictions(index=9)

Time and memory of the learned models are reported in notebook 7, measured over training and over the test evaluation. Reconstructing a frame patch by patch is a cost of this pipeline, not of the models, so it is not attributed to them.